In [1]:
import importlib
import sys
import os

sys.path.append(os.path.abspath("../"))

from utils.gref_pipeline import georef
from gref_pipeline import config

importlib.reload(georef)  # 🔥 Reload to get latest changes
from utils.gref_pipeline.georef import *

print("✅ Module reloaded successfully!")

✅ Module reloaded successfully!


In [2]:
transect = load_transect(config.OUTPUT_FOLDER)
transect.list_files()

cube = transect.select_files(
    [
        # "rad_uhi_20241029_115057_1",
        # "rad_uhi_20241029_115057_2",
        # "rad_uhi_20241029_115057_3",
        # "rad_uhi_20241029_115057_4",
        "rad_uhi_20241029_115057_5",
        # "rad_uhi_20241029_115057_6",
    ]
)
cube.describe()


📋 Files in output (corrected=False):
 1. rad_uhi_20241029_115057_1  | shape=(1333, 968, 210)  | has_georef=True
 2. rad_uhi_20241029_115057_2  | shape=(1946, 968, 210)  | has_georef=True
 3. rad_uhi_20241029_115057_3  | shape=(2227, 968, 210)  | has_georef=True
 4. rad_uhi_20241029_115057_4  | shape=(2463, 968, 210)  | has_georef=True
 5. rad_uhi_20241029_115057_5  | shape=(2462, 968, 210)  | has_georef=True
 6. rad_uhi_20241029_115057_6  | shape=(155, 968, 210)  | has_georef=True
🔄 Rebuilding grids from georef hits for selected files...
   • rad_uhi_20241029_115057_5
  rad_uhi_20241029_115057_5: Using GRIDDED format (T=2462, S=968)
  rad_uhi_20241029_115057_5: Using GRIDDED format (T=2462, S=968)
✅ Combined shapes: X/Y/Z (2462, 968), RGB (2462, 968)
🔄 Loading full hyperspectral cube...
   • Loading rad_uhi_20241029_115057_5...
✅ Combined shapes: X/Y/Z (2462, 968), RGB (2462, 968)
🔄 Loading full hyperspectral cube...
   • Loading rad_uhi_20241029_115057_5...
✅ Loaded full cube: (2462,

In [3]:
cube.apply_illumination_correction_v2()

🔄 Using V2 algorithm (pandas rolling median)
✅ Illumination correction already applied with window=500, strength=1.0
   Loading from disk...
📂 Loading saved illumination correction from 1 files...
   rad_uhi_20241029_115057_5: window=500, strength=1.0
✅ Loaded illumination correction from disk
   rad_uhi_20241029_115057_5: window=500, strength=1.0
✅ Loaded illumination correction from disk


array([[[2.2304249e+00, 1.6764377e+00, 0.0000000e+00, ...,
         0.0000000e+00, 7.0077596e+00, 0.0000000e+00],
        [7.4757171e+00, 8.5974735e-01, 0.0000000e+00, ...,
         0.0000000e+00, 3.0958059e+00, 0.0000000e+00],
        [1.8294926e+00, 2.3438740e+00, 0.0000000e+00, ...,
         0.0000000e+00, 7.1090145e+00, 0.0000000e+00],
        ...,
        [1.7213115e+00, 1.4317783e+00, 0.0000000e+00, ...,
         1.0000000e+00, 2.3279436e-02, 3.7040919e-01],
        [5.8812752e+00, 2.0123754e+00, 0.0000000e+00, ...,
         0.0000000e+00, 1.4956644e+00, 0.0000000e+00],
        [2.3867478e+00, 1.0000000e+00, 0.0000000e+00, ...,
         0.0000000e+00, 1.5736811e+01, 4.5105180e-01]],

       [[2.2304249e+00, 7.7452070e-01, 3.9940205e+00, ...,
         0.0000000e+00, 0.0000000e+00, 9.6023534e-05],
        [1.7471981e+00, 0.0000000e+00, 1.2654871e+00, ...,
         0.0000000e+00, 8.5029960e-01, 6.3767982e-01],
        [1.9216584e+00, 0.0000000e+00, 3.0771544e+00, ...,
         0.000

In [4]:
cube.import_rois("./ROIs/057_5_combined.json")
cube.list_rois()

📂 Imported 6 ROIs from ./ROIs/057_5_combined.json

📂 ROI Collection (6 ROIs):
 1. 'dark spots': 358 pixels
     Slit range:  174 - 740
     Track range: 729 - 1416
 2. 'all bombs': 454 pixels
     Slit range:  244 - 880
     Track range: 593 - 1547
 3. 'sediment': 273 pixels
     Slit range:   14 - 870
     Track range: 577 - 1580
 4. 'training_dark': 364 pixels
     Slit range:   24 - 962
     Track range:  23 - 2420
 5. 'training_sediment': 398 pixels
     Slit range:   31 - 924
     Track range:  51 - 2417
 6. 'training_bombs': 201 pixels
     Slit range:  513 - 794
     Track range: 2136 - 2416


In [5]:
training_rois = [
    "training_dark",
    "training_sediment",
    "training_bombs",
]

validation_rois = [
    "sediment",
    "dark spots",
    "all bombs",
]

In [6]:
# STEP 2: Train SVM with cross-validation on pixels OUTSIDE segment
# 🔥 NEW: Leave-One-Bomb-Out CV strategy ensures all classes in every fold
# - Each fold has exactly ONE bomb group in validation
# - Dark and sediment groups distributed evenly across folds
# - Refuses folds missing any class (reliable scores)
cv_results = cube.train_svm_with_cv(
    training_rois=training_rois,
    segment_start=config.UHI_TRACK_RANGE_5[0],
    segment_end=config.UHI_TRACK_RANGE_5[1],
    wavelength_range=(490, 680),  # ✅ Only use 490-680nm wavelengths
    cv_folds=5,  # Will auto-adjust to number of bomb groups (typically 2)
    use_corrected=True,
    svm_kernel="rbf",
    optimize_params=True,
    quiet=False,
)

# Print summary
print(f"\n" + "=" * 60)
print(f"📊 CROSS-VALIDATION SUMMARY")
print(f"=" * 60)
print(
    f"Accuracy:  {cv_results['cv_mean_metrics']['accuracy_mean']:.3f} ± {cv_results['cv_mean_metrics']['accuracy_std']:.3f}"
)
print(
    f"Precision: {cv_results['cv_mean_metrics']['precision_mean']:.3f} ± {cv_results['cv_mean_metrics']['precision_std']:.3f}"
)
print(
    f"Recall:    {cv_results['cv_mean_metrics']['recall_mean']:.3f} ± {cv_results['cv_mean_metrics']['recall_std']:.3f}"
)
print(
    f"F1 Score:  {cv_results['cv_mean_metrics']['f1_mean']:.3f} ± {cv_results['cv_mean_metrics']['f1_std']:.3f}"
)

print(f"\n🎯 Best hyperparameters:")
print(f"  C = {cv_results['best_params']['C']}")
print(f"  gamma = {cv_results['best_params']['gamma']}")

print(f"\n📍 Training pixels used:")
for class_name, count in cv_results["training_pixels_per_class"].items():
    print(f"  {class_name}: {count} pixels")

print(f"\n🔍 Pixel filtering:")
print(f"  Kept (outside segment): {cv_results['filtered_pixels_outside']}")
print(f"  Rejected (inside segment): {cv_results['filtered_pixels_inside']}")

🤖 SVM TRAINING WITH CROSS-VALIDATION

� Wavelength filtering:
   Range: 490 - 680 nm
   Wavelengths used: 108 (from 210)

�📦 Datacube shape: (2462, 968, 108)
📏 Using data: corrected
🎯 Segment range: tracks 576 to 1566
📍 Training ROIs: ['training_dark', 'training_sediment', 'training_bombs']

🔍 Filtering ROI pixels...
   training_dark: 364 pixels (rejected 0 inside segment)
   training_sediment: 397 pixels (rejected 1 inside segment)
   training_bombs: 201 pixels (rejected 0 inside segment)

✅ Training data: 962 pixels, 108 features
   Features: 108 wavelengths
   Kept (outside segment): 962
   Rejected (inside segment): 1

🔬 Creating spatial groups for CV...
   Settings: closing_radius=3, min_group_size=20
   Sediment grid: 100(slit) × 200(track) px, min=10
   training_dark: 364 pixels → 6 groups via connected_components
      dark#A: 119 pixels
      dark#B: 50 pixels
      dark#C: 37 pixels
      dark#D: 29 pixels
      dark#E: 51 pixels
      dark#F: 78 pixels
      Merged small til

In [7]:
# STEP 3: Classify segment with BASELINE model (balanced weights, no brightness)
classification_results = cube.classify_segment_with_validation(
    segment_start=config.UHI_TRACK_RANGE_5[0],
    segment_end=config.UHI_TRACK_RANGE_5[1],
    validation_rois=validation_rois,
    validation_class_mapping={
        "sediment": "training_sediment",
        "dark spots": "training_dark",
        "all bombs": "training_bombs",
    },
    use_corrected=True,
    save_to_h5=True,
    dataset_name="svm_classification_baseline",
    apply_post_filtering=True,
    post_cc_bomb=True,
    post_cc_dark=True,
    post_cc_sediment=False,
    cc_connectivity=8,
    cc_min_area=20,
    morph_close_radius=1,
    morph_open_radius=1,
    merge_proximity_px=0,
    quiet=False,
)

# Print validation results
if classification_results["validation_metrics"]:
    val_metrics = classification_results["validation_metrics"]

    print(f"\n" + "=" * 60)
    print(f"📈 BASELINE VALIDATION RESULTS")
    print(f"=" * 60)
    print(f"Overall Accuracy: {val_metrics['accuracy']:.3f}")

    print(f"\n📋 Per-Class Metrics:")
    for class_name in classification_results["class_names"]:
        print(f"\n{class_name}:")
        print(f"  Precision: {val_metrics['precision_per_class'][class_name]:.3f}")
        print(f"  Recall:    {val_metrics['recall_per_class'][class_name]:.3f}")
        print(f"  F1 Score:  {val_metrics['f1_per_class'][class_name]:.3f}")
        print(f"  Support:   {val_metrics['support_per_class'][class_name]} pixels")

🎯 SEGMENT CLASSIFICATION WITH VALIDATION

📊 Using wavelength subset: 108 wavelengths

📦 Datacube shape: (2462, 968, 108)
📏 Using data: corrected
🎯 Segment range: tracks 576 to 1566

🔪 Extracting segment...
   Segment shape: (991, 968, 108)

🤖 Classifying 959288 pixels...

📊 Using wavelength subset: 108 wavelengths

📦 Datacube shape: (2462, 968, 108)
📏 Using data: corrected
🎯 Segment range: tracks 576 to 1566

🔪 Extracting segment...
   Segment shape: (991, 968, 108)

🤖 Classifying 959288 pixels...


   Classifying: 100%|████████████████████████████████| 959288/959288 [00:51<00:00, 18795.31pixels/s]



   ✅ Classification complete in 51.07 seconds

📊 Classification distribution (before filtering):
   training_bombs: 1566 pixels (0.2%)
   training_dark: 86448 pixels (9.0%)
   training_sediment: 871274 pixels (90.8%)

🧹 POST-CLASSIFICATION FILTERING
   Classes to filter: ['training_bombs', 'training_dark']
   Connectivity: 8
   Min area: 20 pixels
   Morph close radius: 1 (enabled)
   Morph open radius: 1 (enabled)
   Merge proximity: 0 px (disabled)

   🔍 Filtering training_bombs (bomb)...
      Found 8 connected components
      Kept: 4 components (1443 pixels)
      Removed: 4 components (29 pixels)
      Final: 1443 pixels (123 removed, 7.9%)

   🔍 Filtering training_dark (dark)...
      Found 169 connected components
      Kept: 81 components (85979 pixels)
      Removed: 88 components (753 pixels)
      Final: 85979 pixels (469 removed, 0.5%)

✅ Post-classification filtering complete
      Found 8 connected components
      Kept: 4 components (1443 pixels)
      Removed: 4 compon

## 🔥 STEP 4: Test Custom Class Weights + Brightness Feature

Now train and classify with improved settings:
- **Custom weights**: 2x for bombs (catch more bombs!)
- **Brightness feature**: Mean intensity (490-680nm)

This will be compared with the baseline above.

In [8]:
# STEP 5: Train SVM with custom class weights + brightness feature
custom_weights = {
    "training_bombs": 2.0,  # 2x weight for bombs (improve recall)
    "training_dark": 1.0,
    "training_sediment": 1.0,
}

cv_results_custom = cube.train_svm_with_cv(
    training_rois=training_rois,
    segment_start=config.UHI_TRACK_RANGE_5[0],
    segment_end=config.UHI_TRACK_RANGE_5[1],
    wavelength_range=(490, 680),  # ✅ Only use 490-680nm wavelengths
    cv_folds=5,
    use_corrected=True,
    svm_kernel="rbf",
    optimize_params=True,
    class_weight_dict=custom_weights,  # 🔥 Custom weights (2x for bombs)
    add_brightness_feature=True,  # 🔥 Add brightness (mean intensity)
    quiet=False,
)

# Print summary
print(f"\n" + "=" * 60)
print(f"📊 CROSS-VALIDATION SUMMARY (Custom Weights + Brightness)")
print(f"=" * 60)
print(
    f"Accuracy:  {cv_results_custom['cv_mean_metrics']['accuracy_mean']:.3f} ± {cv_results_custom['cv_mean_metrics']['accuracy_std']:.3f}"
)
print(
    f"Precision: {cv_results_custom['cv_mean_metrics']['precision_mean']:.3f} ± {cv_results_custom['cv_mean_metrics']['precision_std']:.3f}"
)
print(
    f"Recall:    {cv_results_custom['cv_mean_metrics']['recall_mean']:.3f} ± {cv_results_custom['cv_mean_metrics']['recall_std']:.3f}"
)
print(
    f"F1 Score:  {cv_results_custom['cv_mean_metrics']['f1_mean']:.3f} ± {cv_results_custom['cv_mean_metrics']['f1_std']:.3f}"
)

# Compare CV metrics with baseline
print(f"\n📈 CV METRIC CHANGES FROM BASELINE:")
print(
    f"  Accuracy:  {cv_results_custom['cv_mean_metrics']['accuracy_mean'] - cv_results['cv_mean_metrics']['accuracy_mean']:+.3f}"
)
print(
    f"  Precision: {cv_results_custom['cv_mean_metrics']['precision_mean'] - cv_results['cv_mean_metrics']['precision_mean']:+.3f}"
)
print(
    f"  Recall:    {cv_results_custom['cv_mean_metrics']['recall_mean'] - cv_results['cv_mean_metrics']['recall_mean']:+.3f}"
)
print(
    f"  F1:        {cv_results_custom['cv_mean_metrics']['f1_mean'] - cv_results['cv_mean_metrics']['f1_mean']:+.3f}"
)

🤖 SVM TRAINING WITH CROSS-VALIDATION

� Wavelength filtering:
   Range: 490 - 680 nm
   Wavelengths used: 108 (from 210)

�📦 Datacube shape: (2462, 968, 108)
📏 Using data: corrected
🎯 Segment range: tracks 576 to 1566
📍 Training ROIs: ['training_dark', 'training_sediment', 'training_bombs']
💡 Brightness feature: ENABLED (mean of wavelength-filtered spectrum)

🔍 Filtering ROI pixels...
   training_dark: 364 pixels (rejected 0 inside segment)
   training_sediment: 397 pixels (rejected 1 inside segment)
   training_bombs: 201 pixels (rejected 0 inside segment)

💡 Brightness feature added:
   Brightness range: [0.7648, 1.1552]
   Brightness mean: 0.9643 ± 0.0739

✅ Training data: 962 pixels, 109 features
   Features: 108 wavelengths + 1 brightness
   Kept (outside segment): 962
   Rejected (inside segment): 1

🔬 Creating spatial groups for CV...
   Settings: closing_radius=3, min_group_size=20
   Sediment grid: 100(slit) × 200(track) px, min=10
   training_dark: 364 pixels → 6 groups via c

In [ ]:
# STEP 6: Classify segment with CUSTOM model (custom weights + brightness)
classification_results_custom = cube.classify_segment_with_validation(
    segment_start=config.UHI_TRACK_RANGE_5[0],
    segment_end=config.UHI_TRACK_RANGE_5[1],
    validation_rois=validation_rois,
    validation_class_mapping={
        "sediment": "training_sediment",
        "dark spots": "training_dark",
        "all bombs": "training_bombs",
    },
    use_corrected=True,
    save_to_h5=True,
    dataset_name="svm_classification_custom_weights",
    apply_post_filtering=True,
    post_cc_bomb=True,
    post_cc_dark=True,
    post_cc_sediment=False,
    cc_connectivity=8,
    cc_min_area=20,
    morph_close_radius=1,
    morph_open_radius=1,
    merge_proximity_px=0,
    quiet=False,
)

# Print validation results
if classification_results_custom["validation_metrics"]:
    val_metrics_custom = classification_results_custom["validation_metrics"]

    print(f"\n" + "=" * 60)
    print(f"📈 CUSTOM MODEL VALIDATION RESULTS")
    print(f"=" * 60)
    print(f"Overall Accuracy: {val_metrics_custom['accuracy']:.3f}")

    print(f"\n? Per-Class Metrics:")
    for class_name in classification_results_custom["class_names"]:
        print(f"\n{class_name}:")
        print(f"  Precision: {val_metrics_custom['precision_per_class'][class_name]:.3f}")
        print(f"  Recall:    {val_metrics_custom['recall_per_class'][class_name]:.3f}")
        print(f"  F1 Score:  {val_metrics_custom['f1_per_class'][class_name]:.3f}")
        print(f"  Support:   {val_metrics_custom['support_per_class'][class_name]} pixels")
    
    # Compare with baseline
    val_metrics_baseline = classification_results["validation_metrics"]
    
    print(f"\n" + "=" * 60)
    print(f"📊 VALIDATION METRIC CHANGES FROM BASELINE")
    print(f"=" * 60)
    
    print(f"\n{'Metric':<25} {'BASELINE':>12} {'CUSTOM':>12} {'CHANGE':>12}")
    print("-" * 60)
    print(
        f"{'Overall Accuracy':<25} {val_metrics_baseline['accuracy']:>12.3f} {val_metrics_custom['accuracy']:>12.3f} {val_metrics_custom['accuracy']-val_metrics_baseline['accuracy']:>+12.3f}"
    )
    
    print(f"\n📊 Per-Class Changes:")
    for class_name in classification_results_custom["class_names"]:
        print(f"\n{class_name}:")
        print(
            f"  {'Precision:':<23} {val_metrics_baseline['precision_per_class'][class_name]:>12.3f} {val_metrics_custom['precision_per_class'][class_name]:>12.3f} {val_metrics_custom['precision_per_class'][class_name]-val_metrics_baseline['precision_per_class'][class_name]:>+12.3f}"
        )
        print(
            f"  {'Recall:':<23} {val_metrics_baseline['recall_per_class'][class_name]:>12.3f} {val_metrics_custom['recall_per_class'][class_name]:>12.3f} {val_metrics_custom['recall_per_class'][class_name]-val_metrics_baseline['recall_per_class'][class_name]:>+12.3f}"
        )
        print(
            f"  {'F1 Score:':<23} {val_metrics_baseline['f1_per_class'][class_name]:>12.3f} {val_metrics_custom['f1_per_class'][class_name]:>12.3f} {val_metrics_custom['f1_per_class'][class_name]-val_metrics_baseline['f1_per_class'][class_name]:>+12.3f}"
        )

In [ ]:
# STEP 7: Visualize comparison - Baseline vs Custom (side-by-side)
import matplotlib.pyplot as plt


def classification_map_to_roi_collection(
    classification_map, segment_start, class_names
):
    """Convert classification map to ROI collection format for plot_georef."""
    roi_collection = {}
    for class_name in class_names:
        if not class_name:
            continue
        class_mask = classification_map == class_name
        track_indices, slit_indices = np.where(class_mask)
        track_indices_abs = track_indices + segment_start
        pixels = [(slit, track) for slit, track in zip(slit_indices, track_indices_abs)]
        if len(pixels) > 0:
            roi_collection[class_name] = pixels
    return roi_collection


segment_start = config.UHI_TRACK_RANGE_5[0]
segment_end = config.UHI_TRACK_RANGE_5[1]

# Convert classification maps to ROI collections
roi_baseline = classification_map_to_roi_collection(
    classification_results["classification_map"],
    segment_start,
    classification_results["class_names"],
)

roi_custom = classification_map_to_roi_collection(
    classification_results_custom["classification_map"],
    segment_start,
    classification_results_custom["class_names"],
)

# Print pixel counts comparison
print("\n" + "=" * 60)
print("📊 PIXEL COUNTS: Baseline vs Custom")
print("=" * 60)
print(f"{'Class':<25} {'BASELINE':>10} {'CUSTOM':>10} {'CHANGE':>10} {'% Change':>12}")
print("-" * 75)
for class_name in classification_results["class_names"]:
    baseline_count = len(roi_baseline.get(class_name, []))
    custom_count = len(roi_custom.get(class_name, []))
    change = custom_count - baseline_count
    pct_change = (change / baseline_count * 100) if baseline_count > 0 else 0
    print(
        f"{class_name:<25} {baseline_count:>10} {custom_count:>10} {change:>+10} {pct_change:>+11.1f}%"
    )

# Plot BASELINE
print("\n🔍 Plotting BASELINE (balanced weights, no brightness)...")
result_baseline = cube.plot_georef(
    use_corrected=True,
    track_start=segment_start,
    track_end=segment_end,
    figsize=(40, 10),
    roi_collection=roi_baseline,
    roi_marker_size=3,
    roi_legend_loc="outside",
    roi_marker_edgewidth=0,
    roi_legend_markersize=40,
    return_fig=True,
)
fig_baseline, ax_baseline = result_baseline
fig_baseline.suptitle(
    "BASELINE: Balanced Weights + No Brightness", fontsize=16, weight="bold", y=0.98
)
plt.show()

# Plot CUSTOM
print("\n✨ Plotting CUSTOM (2x bombs + brightness)...")
result_custom = cube.plot_georef(
    use_corrected=True,
    track_start=segment_start,
    track_end=segment_end,
    figsize=(40, 10),
    roi_collection=roi_custom,
    roi_marker_size=3,
    roi_legend_loc="outside",
    roi_marker_edgewidth=0,
    roi_legend_markersize=40,
    return_fig=True,
)
fig_custom, ax_custom = result_custom
fig_custom.suptitle(
    "IMPROVED: Custom Weights (2x Bombs) + Brightness Feature",
    fontsize=16,
    weight="bold",
    y=0.98,
)
plt.show()

print("\n✅ Comparison complete! Check if bombs improved above. 🎯")